In [79]:
import numpy as np
import pandas as pd
import optuna
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

# Загрузим данные

- Данные упорядочены по времени
- Задача - предсказать `target`
- Фичи - `feature_i`
- Трейн, валидация и тест уже определены (см. колонку `sample_part`)

In [80]:
df = pd.read_parquet('df.parquet.gzip')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 500000 entries, 22620 to 310653
Columns: 235 entries, date to sample_part
dtypes: category(2), datetime64[ms](1), datetime64[ns](2), float64(220), int64(9), object(1)
memory usage: 893.6+ MB


In [81]:
df.head()

,date,month,quarter,feature_217,feature_66,feature_9,feature_193,feature_15,feature_199,feature_25,...,feature_20,feature_188,feature_71,feature_106,feature_14,feature_92,feature_179,feature_102,target,sample_part
22620,2021-01-01,2021-01-01,2021-01-01,1.180855,0.483118,13.976791,1.135021,43.271792,-46.888875,97.558366,...,23.745198,-32.001093,85.489903,1.454045,-195.725786,3.719083,1,-28.813537,0.0,train
478621,2021-01-01,2021-01-01,2021-01-01,3.427814,0.886778,27.158209,-0.378728,80.087863,-67.242703,105.803406,...,22.902111,-37.436243,71.735619,1.657242,-176.250404,3.277817,1,-45.987721,1.0,train
372254,2021-01-01,2021-01-01,2021-01-01,-38.555018,1.227175,19.893873,0.498787,98.464365,-67.402522,81.858076,...,21.848414,-51.872841,65.363236,1.626874,-178.116798,2.028950,0,-31.074122,1.0,train
2596,2021-01-01,2021-01-01,2021-01-01,-14.666558,0.753129,18.203010,-1.200014,117.331340,-25.557745,90.277266,...,26.134586,-49.703003,65.013659,1.170023,-192.232264,2.218177,1,-36.703897,0.0,train
216892,2021-01-01,2021-01-01,2021-01-01,6.734990,1.129100,11.245920,-0.108955,61.205722,-78.397406,82.637472,...,24.201039,-41.070219,69.986127,1.630795,-200.155528,3.055104,1,-44.672299,1.0,train


In [82]:
TARGET = 'target'
N_FEATURES = 230
features = [f'feature_{i}' for i in range(N_FEATURES)]

# Первичный отбор признаков... (1 балл)

**Задание:** Сначала отсеем совсем уж мусорные признаки.

Воспользовавшись вашим кодом для вычисления `IV` из прошлой домашки, для всех **числовых** фичей вычислите `IV` на 20 бакетах. Отсейте признаки с `IV < 0.005`.

**Hint:** паркет не всегда сохраняет `dtype` колонки. Чтобы проверить признаки на "реальный" тип данных, лучше на всякий случай посмотреть на `nunique` 

In [83]:
# your code here
from scipy.special import logit

def calc_buckets(x : np.ndarray, n_buckets : int) -> np.ndarray:
    """Разбивает массив значений признака x на 
    n_buckets бакетов"""

    x = pd.Series(x).reset_index(drop=True)
    buckets = x.rank(method="dense", pct=True) * n_buckets
    buckets = np.ceil(buckets) - 1   # np.floor дает другой результат для 5.0, 6.0 и т.д.
    buckets = np.array(buckets, dtype=np.int16)

    return buckets
def woe_transform(badrate : float, offset : float) -> float:
    """Считаем WoE для бакета с данным badrate и выборки
    с данным offset."""
    woe = logit(badrate) - offset
    return woe

def calc_IV(
        values : np.ndarray,
        target : np.ndarray,
        buckets : np.ndarray
    ) -> str:
    total = len(target)
    bads = target.sum()
    goods = total - bads
    offset = logit(target.mean())
    IV = sum(
            [
            ((bads_i := target[buckets == i].sum()) / bads - ((m := len(target[buckets == i])) - bads_i) / goods) * 
            woe_transform(bads_i / m, offset)   for i in sorted(set(buckets))
            ]
        ) # всё ещё простите за это...

    return IV

In [ ]:
# В отдельную ячейку, потому что долго.
num_df_columns = [x for x in df.select_dtypes(include=['float64', 'int64']).columns if df[x].nunique() > 50  ]
target_train = df[TARGET][df['sample_part'] == 'train']
iv_arr = {}

In [85]:
for x in num_df_columns:
    val_raw = df[x][df['sample_part'] == 'train']
    val = np.asarray(val_raw)
    buckets = calc_buckets(val, 20)
    iv = calc_IV(val, target_train, buckets)
    iv_arr[x] = iv 

c:\Users\temy2\AppData\Local\Programs\Python\Python310\lib\site-packages\pandas\core\series.py:1031: RuntimeWarning: invalid value encountered in cast
  arr = np.asarray(values, dtype=dtype)


In [86]:
iv_selected_features = [f for f, iv in iv_arr.items() if iv >= 0.005]
len(iv_selected_features), iv_selected_features

(57,
 ['feature_9',
  'feature_15',
  'feature_154',
  'feature_218',
  'feature_96',
  'feature_223',
  'feature_30',
  'feature_225',
  'feature_162',
  'feature_167',
  'feature_109',
  'feature_124',
  'feature_79',
  'feature_213',
  'feature_117',
  'feature_38',
  'feature_24',
  'feature_75',
  'feature_5',
  'feature_56',
  'feature_114',
  'feature_139',
  'feature_65',
  'feature_142',
  'feature_12',
  'feature_35',
  'feature_227',
  'feature_115',
  'feature_155',
  'feature_132',
  'feature_76',
  'feature_97',
  'feature_140',
  'feature_135',
  'feature_2',
  'feature_46',
  'feature_100',
  'feature_111',
  'feature_138',
  'feature_164',
  'feature_98',
  'feature_36',
  'feature_183',
  'feature_27',
  'feature_156',
  'feature_4',
  'feature_32',
  'feature_206',
  'feature_172',
  'feature_70',
  'feature_44',
  'feature_23',
  'feature_170',
  'feature_39',
  'feature_216',
  'feature_94',
  'feature_43'])

**Вопрос**: Почему некорректно сравнивать `IV` у категориальных и числовых фичей?

Вычислите IV для категориальных фичей на `n` бакетах, где `n = min(число категорий фичи, 20)`. Возможно, придётся перекодировать некоторые фичи (*только не OneHot-ом!*)

*Опционально*: примените также и к категориальным фичам предварительный отбор по `IV` с менее строгим порогом

In [87]:
# your code here
cat_df_columns = [x for x in df.select_dtypes(include=['category', 'int64']).columns if df[x].nunique() <= 50 ]
cat_df_columns

['feature_147',
 'feature_144',
 'feature_168',
 'feature_165',
 'feature_31',
 'feature_0',
 'feature_209',
 'feature_222',
 'feature_37',
 'feature_157',
 'feature_179']

In [88]:
iv_selected_cat_features = {}
for y in cat_df_columns:
    val_raw = df[y][df['sample_part'] == 'train']
    val = np.asarray(val_raw)
    buckets = calc_buckets(val, min(20, len(np.unique(val)) ) )
    iv = calc_IV(val, target_train, buckets)
    iv_selected_cat_features[y] = iv 

iv_selected_cat_features = [f for f, iv in iv_selected_cat_features.items() if iv >= 0.001]
len(iv_selected_cat_features), iv_selected_cat_features

(6,
 ['feature_147',
  'feature_144',
  'feature_168',
  'feature_165',
  'feature_31',
  'feature_209'])

**Ответ:** сравнивать IV у категориальных и числовых фичей некорректно, потому что мы теряем какую-то часть информации о непрерывных фичах из-за биннинга, потому что мы для корректного подсчёта IV мы должны относиться к ней, как к дискретной случайной величине, а тогда мы можем очень плохо поделить на бакеты, чтобы получить дискретность, и поэтому получить низкий IV, хотя проблема была не в самой фиче, а в разбиении на бакеты, категориальные фичи же в свою очередь от такой проблемы не страдают, потому что уже дискретны, поэтому для них IV не занижается.

# Бустинг

Используйте код для подбора гиперпараметров и обучения бустинга из предыдущей домашки

In [90]:
# В переменную features_optuna положите список признаков с предыдущего шага
features_optuna = iv_selected_features + iv_selected_cat_features
len(features_optuna)

63

In [89]:
import lightgbm as lgb

In [105]:
import optuna

train_data_df = df[df['sample_part'] == 'train']
valid_data_df = df[df['sample_part'] == 'val']
train_data = lgb.Dataset(train_data_df.drop(columns=[TARGET, 'sample_part', 'date', 'month', 'quarter'], axis = 1), label=train_data_df[TARGET])
valid_data = lgb.Dataset(valid_data_df.drop([TARGET, 'sample_part', 'date', 'month', 'quarter'], axis = 1), label=valid_data_df[TARGET])

X_test = valid_data_df.drop(columns=[TARGET, 'sample_part', 'date', 'month', 'quarter'], axis = 1)

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'n_jobs' : -1,
        # 'random_state': RANDOM_STATE,
        # TODO: задайте гиперпараметры через trial.suggest_*
        'num_leaves': trial.suggest_int('num_leaves', 10, 200),       # TODO
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log = True),    # TODO
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1, step=0.1), # TODO
    }

    # TODO: обучите модель с early stopping и верните AUC на валидации
    model = lgb.train(params=params, train_set=train_data, valid_sets=[valid_data], callbacks=[lgb.early_stopping(stopping_rounds=10)], num_boost_round=250)
    auc = roc_auc_score(valid_data_df[TARGET], model.predict(X_test))
    return auc


# TODO: создайте study и запустите optimize (n_trials >= 50)
study = optuna.create_study(direction = 'maximize')
study.optimize(objective, n_trials=50)

print("Best params:", study.best_params)
print("Best AUC:", study.best_value)

[I 2026-08-17 19:35:25,720] A new study created in memory with name: no-name-6b8b56a9-62f4-4575-b1af-50ed3067deff


Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[74]	valid_0's auc: 0.75349


[I 2026-08-17 19:35:41,814] Trial 0 finished with value: 0.7534895803788246 and parameters: {'num_leaves': 83, 'learning_rate': 0.23647175412177154, 'feature_fraction': 0.8}. Best is trial 0 with value: 0.7534895803788246.


Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[250]	valid_0's auc: 0.70643


[I 2026-08-17 19:36:00,197] Trial 1 finished with value: 0.7064298479844584 and parameters: {'num_leaves': 25, 'learning_rate': 0.009143954251513972, 'feature_fraction': 0.5}. Best is trial 0 with value: 0.7534895803788246.


...

Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[250]	valid_0's auc: 0.753758


[I 2026-08-17 20:12:16,195] Trial 48 finished with value: 0.7537582219524042 and parameters: {'num_leaves': 142, 'learning_rate': 0.014904763120617887, 'feature_fraction': 0.5}. Best is trial 45 with value: 0.7670495735794074.


Training until validation scores don't improve for 10 rounds
Did not meet early stopping. Best iteration is:
[250]	valid_0's auc: 0.760264


[I 2026-08-17 20:13:01,970] Trial 49 finished with value: 0.7602641369596891 and parameters: {'num_leaves': 161, 'learning_rate': 0.020238824117127317, 'feature_fraction': 0.6}. Best is trial 45 with value: 0.7670495735794074.


Best params: {'num_leaves': 200, 'learning_rate': 0.03509908630610692, 'feature_fraction': 0.5}
Best AUC: 0.7670495735794074


## Важность признаков (1 балл)

**Задание:** Постойте гистограмму важности признаков по `split` и `gain` для бустинга из предыдущего пункта.

In [ ]:
#####
# <your code here>
#####


**Вопрос:** Какие выводы можно сделать из полученных графиков?

In [ ]:
# <your answer here>

# Логрег

## plot_metric_time (1 балл)

**Задание:** Реализуйте функцию plot_metric_time, которая будет принимать на вход четыре аргумента, а именно: 
- массив значений фичи, 
- массив значений таргета,
- массив времени (аггрегированного по месяцам или кварталам, то есть `month` или `quarter` в вашей задаче, по дням рисовать не надо),
- метрику, которую необходимо отрисовать (нужно реализовать функцию для двух метрик: `IV` и `roc_auc`)
- число бакетов для вычисления `IV`, если выбрана эта метрика

Можете добваить какие-то ещё аргументы, если вам нужно

Если в фиче есть пропуски, функция должна убирать строки с пропусками из рассмотрения

**Hint**: можно, конечно, реализовать эту функцию через цикл, а можно попробовать разобраться и реализовать её через пандасовские `groupby` -> `apply`, это изящнее и быстрее

In [ ]:
import plotly.graph_objects as go

In [ ]:
def plot_metric_time(
        values: pd.Series,
        target: pd.Series,
        time: pd.Series,
        metric: str = 'IV',
        n_buckets_for_IV: int = 15
) -> go.Figure:

    if metric == 'IV':
        ...
    elif metric == 'roc_auc':
        ...
    else:
        raise NotImplementedError("only 'IV' and 'roc_auc' metrics are implemented")

    # your code here

    plot_title = ...
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x= # месяцы или кварталы, your code here
            y= # аггрегированное значение метрики, your code here
            mode='markers+lines',
            name=# имя фичи, your code here
            ))
    fig.update_layout(
            title_text=plot_title,
            yaxis=dict(title=metric),
            width=1000,
            height=450,
            xaxis=dict(
                domain=[0, .95],
                showgrid=True,
                tickvals=# месяцы или кварталы, your code here
                ),
            margin=dict(l=30, r=30, b=30, t=50),
            )
    fig.show()

    # return fig

In [15]:
# примерно так это должно будет выглядеть
plot_metric_time(df[...], df['target'], df['quarter']) # 

**Задание:** Возьмите **топ-15** фичей получившегося бустинга по важности по `gain`. Отрисуйте для них графики стабильности по `IV` во времени и удалите из рассмотрения те признаки, качество которых деградирует

**NB!** Обращайте внимание на масштаб оси *y!* Иногда признак стабильнее, чем кажется)

Если вам это мешает, можете поменять ось *y*, чтобы она начиналась от нуля в прошлом задании.

In [ ]:
# your code here

На этом моменте предлагаю сделать новый маленький датафрейм, чтобы не "портить" исходный

И проводить все манипуляции с фичами на нём

In [ ]:
df_for_logreg = df[top15features_stable + ['date', 'month', 'quarter', 'target', 'sample_part']]

## Предобработка признаков (3 балла)

**Задание:** Используя функицю woe_line из ДЗ по логрегу, проверьте **числовые** фичи из полученного списка фичей на линейность по WoE на трейн-выборке (если в фиче есть пропуски - дропаем их при отрисовке)

Если фичи нелинейные, **линеаризуйте их**.

Преобразования, которые можно/стоит пробовать:
- клипы (`np.clip`) - зачастую их достаточно
- корень
- квадрат
- логарифм

Если нужно, можно прибавлять к фиче константу или менять её знак

При желании можно "распилить фичу на две половины" (если она немонотонна) и линеаризовать их по отдельности

Однако слишком упираться в линеаризацию фичей не нужно. Если фича ну совсем никак не линеаризуется, в крайнем случае можно её дропнуть или оставить как есть.

При отрисовке можно ограничиться 15-20 бактеами

In [18]:
# your code here

Пришло время заполнить пропуски.

Самый простой вариант для числовых признаков - заполнить их средним значением фичи

**Вопрос**: какие проблемы могут возникнуть при таком заполнении пропусков?

**Задание**: Проверьте, что заполнение средним значением адекватно для тех признаков, где есть пропуски (hint: в нашем датасете - почти всегда адекватно). Если нет, придумайте, как ещё можно заполнить пропуски. Ну и заполните их)

In [19]:
# your code here

Вспомним про категориальные признаки, их нужно закодировать.

**Задание:** Используя `OneHotEncoder` закодируйте категориальные признаки.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# your code here

## Stepwise и обучение логрега (5 баллов)

**Задание (boss)**: Реализуйте Stepwise-алгоритм.

Ваша функция `stepwise` должна принимать на вход:
- Датафрейм со всеми признаками и таргетом
- список с именами рассматриваемых признаков
- строку-имя таргета
- уровни значимости `alpha_in` и `alpha_out`

И возвращать список отобранных признаков.

Во время работы пусть она также выводит, какой признак был включён или исключён и с каким `p-value`

In [ ]:
from scipy.stats.distributions import chi2
from sklearn.metrics import log_loss

In [ ]:
def likelihood_ratio_test(ll_short, ll_long):
    
    """
    вспомогательная функция
    рассчитывает значение p-value для теста отношения правдоподобия
    ll_short — логарифм правдоподобия модели на k переменных
    ll_long — логарифм правдоподобия модели на k+1 переменной

    Returns
    -----
    p-value
    """
    lr = 2 * (ll_short - ll_long)
    return chi2.sf(lr, 1)

In [ ]:
def stepwise(
        df: pd.DataFrame, 
        features: list[str], 
        target: str, 
        alpha_in: float = 0.01, 
        alpha_out:  float = 0.05
        ) -> list[str]:
    

    selected_features = list()

    while True:

        # forward:
        best_feature = None
        if (len(selected_features) < len(features)):


            # your code here


            if best_feature:
                selected_features.append(best_feature)
                print(f"В модель была добавлена переменная {best_feature}, p-value: {round(p_value, 6)}")


        # backward
        worst_feature = None
        if (len(selected_features) > 1):


            # your code here


            if worst_feature:
                selected_features.remove(worst_feature)
                print(f"Из модели была удалена переменная {worst_feature}, p-value: {round(p_value, 6)}")

        if not (best_feature or worst_feature):
            break # но вы можете сформулировать критерий останова по-другому, если вам так будет удобнее писать

    return selected_features

Запустите ваш алгоритм на отобранных фичах со значениями `alpha_in = 0.01`, `alpha_out = 0.02`

Если в степвайз заходят *все переменные*, причём с очень маленькими `p-value` - это неудивительно, ведь вы уже провели серьёзный предварительный отбор фичей.

Чтобы убедиться в коректности работы своего алгоритма, можете попробовать запустить его на каких-нибудь других фичах, откинутых сильно ранее

In [ ]:
selected_features = stepwise(..., ..., ..., alpha_in = 0.01, alpha_out = 0.02)

**Задание**: На получившемся наборе признаков обучим, наконец, логрег!

Для обучения можно использовать трейн + валидацию вместе, либо просто трейн

*Не забудьте отскалировать фичи*

In [ ]:
from sklearn.preprocessing import StandardScaler # Или другой скейлер
from sklearn.linear_model import LogisticRegression
from scipy.special import logit

In [ ]:
scaler = StandardScaler()
scaler.fit(...)
# your code here
logreg = LogisticRegression(penalty=None)
logreg.fit(...)
df_for_logreg['logreg'] = logreg.predict_proba(...)[:, 1]
df_for_logreg['logit_logreg'] = logit(df_for_logreg['logreg'])

**Задание:** Оцените качество получившегося логрега **на тестовой выборке** по метрике `roc_auc`. 

**Ваша задача - побить** `threshold_auc = 0.622`. Если этого сделать ну совсем не получается, попробуйте добавить в модель ещё переменных (но должно получаться :)

**Также:**

- Постройте графики линейности по WoE получившегося логрега на трейне и тестовой выборке (оценивайте линейность для `logit_logreg`, а не для вероятности!)
- Постройте график стабильности `roc_auc` во времени для получившегося логрега

In [ ]:
# your code here

**Задание:** Визуализируйте важность фичей полученной линейной модели.

In [ ]:
# your code here

# Бонусная часть (4 балла)

В разделе про бустинг мы обучали `LGBMClassifier` на довольно большом наборе фичей. Их количество можно сократить, при этом не теряя в качестве модели.

За начальный набор признаков можно взять все признаки (все 230) или признаки после отбора по IV.

**Ваша задача:** Отобрать признаки, подобрать оптимальные гиперпараметры и обучить `LGBMClassifier`.

Задание творческое) Можно использовать любые методы отбора признаков / оптимизации гиперпараметров.

**Чем меньше признаков, без ухудшения качества модели – тем лучше.**

In [ ]:
# your code here

Идеи для отбора признаков:

- Воспользоваться методами из модуля `sklearn.feature_selection` (точно можно попробовать RFE). Документация https://scikit-learn.org/stable/modules/feature_selection.html
<!-- - С помощью `BorutaPy` из библиотеки `boruta` (https://towardsdatascience.com/simple-example-using-boruta-feature-selection-in-python-8b96925d5d7a – статья может помочь разобраться с запуском алгоритма (мб не откроется без vpn)) -->
- permutation importance и elimination по нему (https://scikit-learn.org/stable/modules/permutation_importance.html)
- попробовать использовать shap-values

In [ ]:
# your answere here